# Regresão Logística
A regressão logística é um algoritmo de classificação supervisionada. Apesar do nome "regressão", ela é usada para prever categorias, não valores contínuos.

O modelo usa uma função chamada Sigmoid que **"espreme"** qualquer valor em um número entre 0 e 1, interpretado como probabilidade:
- Se a probabilidade for ≥ 0.5 → prediz 1 (sobreviveu)
- Se for < 0.5 → prediz 0 (não sobreviveu)

In [2]:
import numpy as np
import csv

In [3]:
# 1. LEITURA DO CSV

def load_data(filename):
    X = []
    y = []

    with open(filename, "r", encoding='utf-8') as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Target 
            survived = row["survived"]
            if survived == "":
                continue

            y.append(int(survived))

            # Features selecionados
            # Vamos usar:
            # pclass, sex, age, sibsp, parch, fare

            pclass = float(row["pclass"]) if row["pclass"] != "" else 0

            # COnvertendo sexo
            sex = 1 if row['sex'] == 'female' else 0

            age = float(row['age']) if row['age'] != "" else 0

            sibsp = float(row['sibsp']) if row['sibsp'] != "" else 0
            parch = float(row["parch"]) if row['parch'] != "" else 0
            fare = float(row['fare']) if row['fare'] != "" else 0

            X.append([pclass, sex, age, sibsp, parch, fare])

    return np.array(X), np.array(y).reshape(-1,1)



In [4]:
# 2. TRATAMENTO DE DADOS

def fill_missing_afe(X):
    ages = X[:, 2]
    mean_age = np.mean(ages[ages != -1])
    
    X[:,2] = np.where(ages == -1, mean_age, ages)
    return X

def normalize(X):    
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / (std + 1e-8)

def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

In [5]:
# 3. FUNÇÕES DO MODELO

def sigmoid(z):
    return 1/ (1 + np.exp(-z))

def compute_const(X, y, beta):
    m = len(y)
    h = sigmoid(X @ beta)
    epsilon = 1e-8 

    cost = -(1/m) * np.sum(
        y * np.log(h+ epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )
    return cost

def gradient_descent(X, y, beta, lr, epochs):
    m = len(y)

    for i in range(epochs):
        h = sigmoid (X @ beta)
        gradient = (1/m) * (X.T @ (h - y))

        beta = beta - lr * gradient

        if i % 100 ==0:
            print(f"Epoch {i} - Cost: {compute_const(X, y, beta):.4f}")

    return beta

In [6]:
# 4. TREINAMENTO

def train(X, y):
    X = fill_missing_afe(X)
    X = normalize(X)
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, lr=0.01, epochs=2000)

    return beta, X

In [7]:
# 5. PREDIÇÃO E CALCULO DA MATRIZ DE CONFUSÃO

def predict(X, beta):
    probs = sigmoid(X @ beta)

    return (probs >= 0.5).astype(int)


def confusion_matrix(y_true, y_pred):
    # Achatar os arrays para garantir que tenha apenas 1 dimensão
    y_t = y_true.flatten()
    y_p = y_pred.flatten()

    # O operador '&' faz a comparação bit a bit (element-wise) nos arrays do numpy
    # Faz AND lógico, elemento por elemento de cada array.
    # Por Exemplo:
    # y_t = np.array([1, 0, 1, 1])
    # cond1 = (y_t == 1)
    # REsultado: [True, False, True, True]
    VP = np.sum((y_t == 1) & (y_p == 1))
    VN = np.sum((y_t == 0) & (y_p == 0))
    FP = np.sum((y_t == 0) & (y_p == 1))
    FN = np.sum((y_t == 1) & (y_p == 0))

    return VP, VN, FP, FN


def calculate_metrics(VP, VN, FP, FN):
    acc = (VP + VN) / (VP + VN + FP + FN)
    prec = VP / (VP + FP)
    rec = VP / (VP + FN)
    spec = VN / (VN + FP)
    f1 = 2 * (prec * rec) / (prec + rec)

    return acc, prec, rec, spec, f1


In [8]:
# 6. SPLIT TREINO / TESTE

def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    train_idx = indices[:split]
    test_idx = indices[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [9]:
# 7. EXECUÇÃO

X, y = load_data('titanic.csv')

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train)

# Processar teste com MESMAS tranformações 
X_test = fill_missing_afe(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

#Vetor de Predições
y_pred = predict(X_test, beta)

# Métrica da Matriz de Confusão
VP, VN, FP, FN = confusion_matrix(y_test, y_pred)

print('\n' + '='*30)
print(' MATRIZ DE CONFUSÃO')
print('='*30)
print(f'Verdadeiros Positivos (VP): {VP}')
print(f'Verdadeiros Negativos (VN): {VN}')
print(f'Falsos Positivos (FP)       {FP}')
print(f'Falsos Negativos (FN):      {FN}')

# 2. Calcular e exibir as Métricas
acc, prec, rec, spec, f1 = calculate_metrics(VP, VN, FP, FN)

print('\n' + '='*30)
print(' MÉTRICA DE AVALIAÇÃO')
print('='*30)
print(f'Acurácia:       {acc:.4f} ({(acc*100):.1f}%)')
print(f'Precisão        {prec:.4f} ({(prec*100):.1f}%)')

Epoch 0 - Cost: 0.6918
Epoch 100 - Cost: 0.5977
Epoch 200 - Cost: 0.5477
Epoch 300 - Cost: 0.5184
Epoch 400 - Cost: 0.4999
Epoch 500 - Cost: 0.4876
Epoch 600 - Cost: 0.4791
Epoch 700 - Cost: 0.4731
Epoch 800 - Cost: 0.4686
Epoch 900 - Cost: 0.4653
Epoch 1000 - Cost: 0.4628
Epoch 1100 - Cost: 0.4609
Epoch 1200 - Cost: 0.4594
Epoch 1300 - Cost: 0.4582
Epoch 1400 - Cost: 0.4573
Epoch 1500 - Cost: 0.4565
Epoch 1600 - Cost: 0.4559
Epoch 1700 - Cost: 0.4554
Epoch 1800 - Cost: 0.4549
Epoch 1900 - Cost: 0.4546

 MATRIZ DE CONFUSÃO
Verdadeiros Positivos (VP): 68
Verdadeiros Negativos (VN): 123
Falsos Positivos (FP)       32
Falsos Negativos (FN):      39

 MÉTRICA DE AVALIAÇÃO
Acurácia:       0.7290 (72.9%)
Precisão        0.6800 (68.0%)


---
#### **VP, VN, FP, FN — Matriz de Confusão** 

>É uma tabela usada para avaliar o desempenho de modelos de classificação em machine learning. Ela mostra, de forma organizada, quantas previsões o modelo acertou e errou, separadas por classe.

Após o modelo fazer as predições, compara-se com a resposta real:

| |Predito: 1 |Predito: 0| 
| :-- | :-- | :-- | 
|Real: 1 |✅ VP | ❌ FN |
|Real: 0 | ❌ FP | ✅ VN |


- **VP (Verdadeiro Positivo)** — o passageiro sobreviveu e o modelo acertou dizendo que sim
- **VN (Verdadeiro Negativo)** — o passageiro não sobreviveu e o modelo acertou dizendo que não
- **FP (Falso Positivo)** — o passageiro não sobreviveu, mas o modelo errou dizendo que sim (alarme falso)
- **FN (Falso Negativo)** — o passageiro sobreviveu, mas o modelo errou dizendo que não (o pior erro em casos críticos)
---

In [13]:
# Durante o treino, salvar os parâmetros:
def train(X, y):
    X = fill_missing_afe(X)
    X, mean, std = normalize(X)   # salva mean e std
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))
    beta = gradient_descent(X, y, beta, lr=0.01, epochs=2000)

    return beta, X, mean, std   # retorna junto

# Normalize agora aceita mean e std externos:
def normalize(X, mean=None, std=None):
    if mean is None:
        mean = np.mean(X, axis=0)
        std  = np.std(X, axis=0)
    return (X - mean) / (std + 1e-8), mean, std


# Na função de previsão, usar o mean e std do treino:
def prever_passageiro(beta, mean, std, pclass, sex, age, sibsp, parch, fare):
    x = np.array([[pclass,
                   1 if sex == 'female' else 0,
                   age, sibsp, parch, fare]])

    x, _, _ = normalize(x, mean, std)  # usa parâmetros do treino
    x = add_bias(x)

    prob = sigmoid(x @ beta)
    resultado = "Sobreviveu" if prob >= 0.5 else "Não sobreviveu"
    print(f"Probabilidade de sobrevivência: {prob[0][0]:.2%}")
    print(f"Resultado: {resultado}")


# Execução:
beta, X_train_processed, mean, std = train(X_train, y_train)

prever_passageiro(beta, mean, std, pclass=1, sex='male', age=5, sibsp=0, parch=2, fare=151.5500)


Epoch 0 - Cost: 0.6918
Epoch 100 - Cost: 0.5977
Epoch 200 - Cost: 0.5477
Epoch 300 - Cost: 0.5184
Epoch 400 - Cost: 0.4999
Epoch 500 - Cost: 0.4876
Epoch 600 - Cost: 0.4791
Epoch 700 - Cost: 0.4731
Epoch 800 - Cost: 0.4686
Epoch 900 - Cost: 0.4653
Epoch 1000 - Cost: 0.4628
Epoch 1100 - Cost: 0.4609
Epoch 1200 - Cost: 0.4594
Epoch 1300 - Cost: 0.4582
Epoch 1400 - Cost: 0.4573
Epoch 1500 - Cost: 0.4565
Epoch 1600 - Cost: 0.4559
Epoch 1700 - Cost: 0.4554
Epoch 1800 - Cost: 0.4549
Epoch 1900 - Cost: 0.4546
Probabilidade de sobrevivência: 56.66%
Resultado: Sobreviveu
